# AuraGateway v2 — Reconcile-Balance Action-Extraction Canary v1

**Authorized scope:** 12 synthetic first-attempt requests  
**Model:** `Qwen/Qwen2.5-0.5B-Instruct` at the frozen revision  
**Worker:** `worker_1` on GPU 0  
**Retries / repairs / replacements:** 0  
**Cache measurement:** out of scope  
**Full measured benchmark:** unauthorized

Run cells in order. Do not start the worker or issue model requests unless every
artifact, source, authorization, model, wheel, and runtime preflight passes.


In [ ]:
# Cell 01 — Standard Library, Frozen Identities, and Paths

from __future__ import annotations

import hashlib
import importlib
import importlib.metadata
import json
import os
import re
import shutil
import signal
import socket
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

PACKAGE_ZIP_NAME = "auragateway-local-abc-action-extraction-notebook-v1.zip"
NOTEBOOK_ARCHIVE_PATH = (
    "notebooks/kaggle/auragateway_v2_reconcile_balance_action_extraction_canary_v1.ipynb"
)
BINDING_ARCHIVE_PATH = (
    "benchmarks/local_abc/reconcile_balance_extraction_canary_notebook_binding_v1.json"
)
NOTEBOOK_FILENAME = "auragateway_v2_reconcile_balance_action_extraction_canary_v1.ipynb"
BINDING_FILENAME = "reconcile_balance_extraction_canary_notebook_binding_v1.json"

REPOSITORY_URL = "https://github.com/kablewithak/auragateway.git"
AUTHORIZATION_MERGE_COMMIT = "0619867a7acbee5e4c5b639963cf1046cbf36809"
AUTHORIZATION_FINGERPRINT = "9efe45c37b3223b6f01bd55e6471a1c487b5115ba6260b77bd3a6ff2219933a9"
AUTHORIZATION_FILE_BLOB_SHA = "f435e78ccec67b4f361b7b766aa8fc4f663f1397"
HARNESS_MERGE_COMMIT = "42ef2e6e7d268d0213c2f3a4a48aa536c04eba59"
IMPLEMENTATION_COMMIT = "0e4f761de11c85ccf40d234e93a5b2d974590612"
CASE_MANIFEST_SHA256 = "babfd460048784991041957fc50e29853d6caa29ba195207bd8f2ad1088bbbf5"
EVALUATION_PLAN_SHA256 = "53a9dc8f3418b4df86151ad9763d44ddd16179ed5d4ca7ac505c3b2f7e401b62"
PROMPT_POLICY_SHA256 = "5f5415b907552bad09dfe16f0537dac0834fd42493579d91090d1b416daa2ec9"
ACTION_SCHEMA_SHA256 = "923c7fb8c5abadf80c65e55516330e7ec48bd5147ec24662a8cc5dbeed0b76a7"
VLLM_WHEEL_SHA256 = "9e206f370c934a2d4b6b1f05d3d09708d344e05d80260189ef19f60755709431"
MODEL_REPOSITORY = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
MODEL_MANIFEST_SHA256 = "b5c53c05aa258cf85b8ac7c1f41ec81aaa6d9d66a656d32f7271bf5d4c9b8daa"
SERVED_MODEL_NAME = MODEL_REPOSITORY

WORK_ROOT = Path("/kaggle/working/auragateway_action_extraction_canary_v1")
REPO_DIR = WORK_ROOT / "auragateway"
MODEL_DIR = WORK_ROOT / "model"
EVIDENCE_DIR = WORK_ROOT / "evidence"
EVIDENCE_ARCHIVE = Path(
    "/kaggle/working/auragateway-reconcile-balance-action-extraction-canary-evidence-v1.zip"
)
WORKER_PORT = 8001
WORKER_LOG = EVIDENCE_DIR / "worker_1.log"

EXPECTED_MODEL_FILES = [
    {
        "path": ".gitattributes",
        "sha256": "11ad7efa24975ee4b0c3c3a38ed18737f0658a5f75a0a96787b576a78a023361",
        "size_bytes": 1519,
    },
    {
        "path": "LICENSE",
        "sha256": "832dd9e00a68dd83b3c3fb9f5588dad7dcf337a0db50f7d9483f310cd292e92e",
        "size_bytes": 11343,
    },
    {
        "path": "README.md",
        "sha256": "b19c806a904db6dc878a0462e70b551f6b7ac78dfbb88c2eb966ca2b9109ae15",
        "size_bytes": 4917,
    },
    {
        "path": "config.json",
        "sha256": "18e18afcaccafade98daf13a54092927904649e1dd4eba8299ab717d5d94ff45",
        "size_bytes": 659,
    },
    {
        "path": "generation_config.json",
        "sha256": "e558847a8b4402616f1273797b015104dc266fe4b520056fca88823ba8f8ebe6",
        "size_bytes": 242,
    },
    {
        "path": "merges.txt",
        "sha256": "599bab54075088774b1733fde865d5bd747cbcc7a547c5bc12610e874e26f5e3",
        "size_bytes": 1671839,
    },
    {
        "path": "model.safetensors",
        "sha256": "fdf756fa7fcbe7404d5c60e26bff1a0c8b8aa1f72ced49e7dd0210fe288fb7fe",
        "size_bytes": 988097824,
    },
    {
        "path": "tokenizer.json",
        "sha256": "c0382117ea329cdf097041132f6d735924b697924d6f6fc3945713e96ce87539",
        "size_bytes": 7031645,
    },
    {
        "path": "tokenizer_config.json",
        "sha256": "5b5d4f65d0acd3b2d56a35b56d374a36cbc1c8fa5cf3b3febbbfabf22f359583",
        "size_bytes": 7305,
    },
    {
        "path": "vocab.json",
        "sha256": "ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910",
        "size_bytes": 2776833,
    },
]

ARTIFACT_PREFLIGHT_QUALIFIED = False
SOURCE_PREFLIGHT_QUALIFIED = False
RUNTIME_PREFLIGHT_QUALIFIED = False
MODEL_PREFLIGHT_QUALIFIED = False
EXECUTION_PREFLIGHT_QUALIFIED = False

WORK_ROOT.mkdir(parents=True, exist_ok=True)
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

print("frozen identities and paths loaded")

In [ ]:
# Cell 02 — Attached Artifact and Exact Notebook Hash Preflight


def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_json(value: object) -> str:
    return json.dumps(
        value,
        ensure_ascii=True,
        separators=(",", ":"),
        sort_keys=True,
    )


package_candidates = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE_ZIP_NAME}"))
if len(package_candidates) != 1:
    raise RuntimeError(
        f"expected exactly one attached {PACKAGE_ZIP_NAME}, observed {len(package_candidates)}"
    )

PACKAGE_ZIP_PATH = package_candidates[0]
with zipfile.ZipFile(PACKAGE_ZIP_PATH) as archive:
    if archive.testzip() is not None:
        raise RuntimeError("attached notebook package failed ZIP integrity")
    notebook_source_bytes = archive.read(NOTEBOOK_ARCHIVE_PATH)
    binding_bytes = archive.read(BINDING_ARCHIVE_PATH)

binding = json.loads(binding_bytes.decode("utf-8"))
if not isinstance(binding, dict):
    raise RuntimeError("notebook binding must be one JSON object")

SOURCE_NOTEBOOK_SHA256 = sha256_bytes(notebook_source_bytes)
if binding.get("notebook_sha256") != SOURCE_NOTEBOOK_SHA256:
    raise RuntimeError("attached notebook source SHA-256 mismatch")
if binding.get("notebook_filename") != NOTEBOOK_FILENAME:
    raise RuntimeError("notebook filename binding drifted")
if binding.get("authorization_fingerprint") != AUTHORIZATION_FINGERPRINT:
    raise RuntimeError("authorization fingerprint binding drifted")
if binding.get("authorization_merge_commit") != AUTHORIZATION_MERGE_COMMIT:
    raise RuntimeError("authorization merge-commit binding drifted")
if binding.get("case_count") != 12:
    raise RuntimeError("notebook binding case count drifted")
if binding.get("full_measured_rerun_authorized") is not False:
    raise RuntimeError("notebook binding cannot authorize the full benchmark")

ARTIFACT_PREFLIGHT_QUALIFIED = True
print(
    canonical_json(
        {
            "status": "ARTIFACT_PREFLIGHT_QUALIFIED",
            "package_filename": PACKAGE_ZIP_PATH.name,
            "notebook_sha256": SOURCE_NOTEBOOK_SHA256,
            "authorization_merge_commit": AUTHORIZATION_MERGE_COMMIT,
            "case_count": 12,
        }
    )
)

In [ ]:
# Cell 03 — Exact Repository Checkout

if not ARTIFACT_PREFLIGHT_QUALIFIED:
    raise RuntimeError("artifact preflight must pass before repository checkout")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--quiet", "--no-checkout", REPOSITORY_URL, str(REPO_DIR)],
    check=True,
)
subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "checkout",
        "--quiet",
        "--detach",
        AUTHORIZATION_MERGE_COMMIT,
    ],
    check=True,
)
observed_commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()
if observed_commit != AUTHORIZATION_MERGE_COMMIT:
    raise RuntimeError("repository checkout did not bind the authorization merge")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-deps",
        "--editable",
        str(REPO_DIR),
    ],
    check=True,
)

print(
    canonical_json(
        {
            "status": "REPOSITORY_CHECKOUT_QUALIFIED",
            "repository": "kablewithak/auragateway",
            "commit": observed_commit,
        }
    )
)

In [ ]:
# Cell 04 — Repository Authorization and Fixed-Scope Preflight

from auragateway.local_abc.action_extraction_authorization import (
    ActionExtractionModelBinding,
    ActionExtractionRuntimeBinding,
    build_action_extraction_notebook_binding,
    fixed_action_extraction_case_ids,
    load_action_extraction_authorization_package,
)
from auragateway.local_abc.action_extraction_eval import (
    build_action_extraction_evaluation_report,
    build_reconcile_balance_extraction_response_format,
    evaluate_reconcile_balance_extraction,
    render_reconcile_balance_extraction_prompt,
)

MANIFEST_PATH = REPO_DIR / "benchmarks/local_abc/reconcile_balance_extraction_eval_cases_v1.json"
PLAN_PATH = REPO_DIR / "benchmarks/local_abc/reconcile_balance_extraction_eval_plan_v1.json"
AUTHORIZATION_PATH = (
    REPO_DIR / "benchmarks/local_abc/" / "reconcile_balance_extraction_canary_authorization_v1.json"
)

authorization_package = load_action_extraction_authorization_package(
    manifest_path=MANIFEST_PATH,
    plan_path=PLAN_PATH,
    authorization_path=AUTHORIZATION_PATH,
)
authorization = authorization_package.authorization
evaluation_package = authorization_package.evaluation
manifest = evaluation_package.manifest
plan = evaluation_package.plan

if authorization.fingerprint() != AUTHORIZATION_FINGERPRINT:
    raise RuntimeError("repository authorization fingerprint mismatch")
if authorization.harness_merge_commit != HARNESS_MERGE_COMMIT:
    raise RuntimeError("harness merge-commit binding drifted")
if authorization.implementation_commit != IMPLEMENTATION_COMMIT:
    raise RuntimeError("implementation commit binding drifted")
if authorization.case_manifest_sha256 != CASE_MANIFEST_SHA256:
    raise RuntimeError("case-manifest fingerprint drifted")
if authorization.evaluation_plan_sha256 != EVALUATION_PLAN_SHA256:
    raise RuntimeError("evaluation-plan fingerprint drifted")
if authorization.prompt_policy_sha256 != PROMPT_POLICY_SHA256:
    raise RuntimeError("prompt-policy fingerprint drifted")
if authorization.action_schema_sha256 != ACTION_SCHEMA_SHA256:
    raise RuntimeError("action-schema fingerprint drifted")

FIXED_CASE_IDS = fixed_action_extraction_case_ids(manifest)
if authorization.selected_case_ids != FIXED_CASE_IDS:
    raise RuntimeError("fixed case order drifted")
if len(FIXED_CASE_IDS) != 12:
    raise RuntimeError("fixed case count must remain 12")
if authorization.stop_policy.request_attempts_per_case != 1:
    raise RuntimeError("authorization must remain one attempt per case")
if authorization.stop_policy.hidden_retry_count != 0:
    raise RuntimeError("hidden retries are prohibited")
if authorization.full_measured_rerun_authorized:
    raise RuntimeError("full measured execution remains blocked")
if authorization.cache_measurement_in_scope:
    raise RuntimeError("cache measurement is out of scope")

SOURCE_PREFLIGHT_QUALIFIED = True
print(
    canonical_json(
        {
            "status": "SOURCE_PREFLIGHT_QUALIFIED",
            "authorization_fingerprint": authorization.fingerprint(),
            "case_count": len(FIXED_CASE_IDS),
            "first_case": FIXED_CASE_IDS[0],
            "last_case": FIXED_CASE_IDS[-1],
            "cache_measurement_in_scope": False,
        }
    )
)

In [ ]:
# Cell 05 — Exact vLLM Wheel Installation

if not SOURCE_PREFLIGHT_QUALIFIED:
    raise RuntimeError("source preflight must pass before runtime setup")

wheel_matches: list[Path] = []
for wheel_path in sorted(Path("/kaggle/input").glob("**/*.whl")):
    if sha256_file(wheel_path) == VLLM_WHEEL_SHA256:
        wheel_matches.append(wheel_path)

if len(wheel_matches) != 1:
    raise RuntimeError("expected exactly one attached vLLM wheel with the authorized SHA-256")

VLLM_WHEEL_PATH = wheel_matches[0]
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-deps",
        "--force-reinstall",
        str(VLLM_WHEEL_PATH),
    ],
    check=True,
)
importlib.invalidate_caches()

print(
    canonical_json(
        {
            "status": "VLLM_WHEEL_QUALIFIED",
            "wheel_filename": VLLM_WHEEL_PATH.name,
            "wheel_sha256": sha256_file(VLLM_WHEEL_PATH),
        }
    )
)

In [ ]:
# Cell 06 — GPU, PyTorch, CUDA, and vLLM Runtime Preflight

import torch
import vllm

gpu_count = torch.cuda.device_count()
gpu_names = tuple(torch.cuda.get_device_name(index) for index in range(gpu_count))
compute_capabilities = tuple(
    ".".join(str(value) for value in torch.cuda.get_device_capability(index))
    for index in range(gpu_count)
)
vllm_distribution_version = importlib.metadata.version("vllm")
vllm_module_version = str(vllm.__version__)

runtime_payload = {
    "gpu_count": gpu_count,
    "gpu_name": gpu_names[0] if gpu_names else "",
    "compute_capability": (compute_capabilities[0] if compute_capabilities else ""),
    "torch_version": str(torch.__version__),
    "torch_cuda_version": str(torch.version.cuda),
    "vllm_module_version": vllm_module_version,
    "vllm_distribution_version": vllm_distribution_version,
    "vllm_wheel_sha256": sha256_file(VLLM_WHEEL_PATH),
}
observed_runtime = ActionExtractionRuntimeBinding.model_validate(runtime_payload)

if gpu_names != ("Tesla T4", "Tesla T4"):
    raise RuntimeError(f"expected two Tesla T4 GPUs, observed {gpu_names}")
if compute_capabilities != ("7.5", "7.5"):
    raise RuntimeError(
        f"expected compute capability 7.5 on both GPUs, observed {compute_capabilities}"
    )
if observed_runtime != authorization.runtime:
    raise RuntimeError("observed runtime identity does not match authorization")

RUNTIME_PREFLIGHT_QUALIFIED = True
print(
    canonical_json(
        {
            "status": "RUNTIME_PREFLIGHT_QUALIFIED",
            "gpu_count": gpu_count,
            "gpu_names": gpu_names,
            "compute_capabilities": compute_capabilities,
            "torch_version": str(torch.__version__),
            "cuda_version": str(torch.version.cuda),
            "vllm_module_version": vllm_module_version,
            "vllm_distribution_version": vllm_distribution_version,
        }
    )
)

In [ ]:
# Cell 07 — Exact Model Snapshot Acquisition and Manifest Preflight

if not RUNTIME_PREFLIGHT_QUALIFIED:
    raise RuntimeError("runtime preflight must pass before model acquisition")

from huggingface_hub import snapshot_download

if MODEL_DIR.exists():
    shutil.rmtree(MODEL_DIR)

snapshot_download(
    repo_id=MODEL_REPOSITORY,
    revision=MODEL_REVISION,
    local_dir=MODEL_DIR,
    allow_patterns=[item["path"] for item in EXPECTED_MODEL_FILES],
)

observed_model_manifest: list[dict[str, object]] = []
for expected in EXPECTED_MODEL_FILES:
    model_path = MODEL_DIR / str(expected["path"])
    if not model_path.is_file():
        raise RuntimeError(f"model snapshot file missing: {expected['path']}")
    observed = {
        "path": expected["path"],
        "sha256": sha256_file(model_path),
        "size_bytes": model_path.stat().st_size,
    }
    if observed != expected:
        raise RuntimeError(f"model snapshot mismatch: {expected['path']}")
    observed_model_manifest.append(observed)

observed_manifest_sha256 = hashlib.sha256(
    canonical_json(observed_model_manifest).encode("utf-8")
).hexdigest()
if observed_manifest_sha256 != MODEL_MANIFEST_SHA256:
    raise RuntimeError("model snapshot manifest fingerprint mismatch")

observed_model = ActionExtractionModelBinding(
    repository=MODEL_REPOSITORY,
    revision=MODEL_REVISION,
    tokenizer_revision=MODEL_REVISION,
    model_manifest_sha256=observed_manifest_sha256,
    config_sha256=observed_model_manifest[3]["sha256"],
    generation_config_sha256=observed_model_manifest[4]["sha256"],
    tokenizer_json_sha256=observed_model_manifest[7]["sha256"],
    tokenizer_config_sha256=observed_model_manifest[8]["sha256"],
)
if observed_model != authorization.model:
    raise RuntimeError("observed model identity does not match authorization")

(EVIDENCE_DIR / authorization.evidence.model_snapshot_filename).write_text(
    canonical_json(observed_model_manifest) + "\n",
    encoding="utf-8",
)

MODEL_PREFLIGHT_QUALIFIED = True
print(
    canonical_json(
        {
            "status": "MODEL_PREFLIGHT_QUALIFIED",
            "repository": MODEL_REPOSITORY,
            "revision": MODEL_REVISION,
            "manifest_sha256": observed_manifest_sha256,
            "file_count": len(observed_model_manifest),
        }
    )
)

In [ ]:
# Cell 08 — Notebook Binding and Hash-Only Schedule Preflight

if not MODEL_PREFLIGHT_QUALIFIED:
    raise RuntimeError("model preflight must pass before execution binding")

notebook_runtime_binding = build_action_extraction_notebook_binding(
    package=authorization_package,
    authorization_merge_commit=AUTHORIZATION_MERGE_COMMIT,
    notebook_sha256=SOURCE_NOTEBOOK_SHA256,
    observed_model=observed_model,
    observed_runtime=observed_runtime,
)
if notebook_runtime_binding.authorization_sha256 != AUTHORIZATION_FINGERPRINT:
    raise RuntimeError("runtime notebook authorization fingerprint drifted")
if notebook_runtime_binding.authorization_merge_commit != (AUTHORIZATION_MERGE_COMMIT):
    raise RuntimeError("runtime notebook merge-commit binding drifted")
if notebook_runtime_binding.notebook_sha256 != SOURCE_NOTEBOOK_SHA256:
    raise RuntimeError("runtime notebook SHA-256 binding drifted")

response_format = build_reconcile_balance_extraction_response_format()
schedule_records: list[dict[str, object]] = []
for sequence, case in enumerate(manifest.accepted_cases, start=1):
    transient_prompt = render_reconcile_balance_extraction_prompt(case)
    transient_body = {
        "model": SERVED_MODEL_NAME,
        "messages": [{"role": "user", "content": transient_prompt}],
        "response_format": response_format,
        "temperature": float(authorization.decoding.temperature),
        "top_p": float(authorization.decoding.top_p),
        "seed": authorization.decoding.seed,
        "max_tokens": authorization.decoding.max_output_tokens,
        "stream": authorization.decoding.stream,
    }
    schedule_records.append(
        {
            "schema_version": "1.0.0",
            "sequence": sequence,
            "eval_case_id": case.eval_case_id,
            "expected_action_sha256": case.expected_action_sha256,
            "expected_answer": case.expected_answer,
            "prompt_sha256": case.prompt_sha256,
            "rendered_prompt_sha256": hashlib.sha256(transient_prompt.encode("utf-8")).hexdigest(),
            "rendered_prompt_character_count": len(transient_prompt),
            "request_body_sha256": hashlib.sha256(
                canonical_json(transient_body).encode("utf-8")
            ).hexdigest(),
            "worker_id": "worker_1",
            "attempt_limit": 1,
        }
    )

if tuple(item["eval_case_id"] for item in schedule_records) != FIXED_CASE_IDS:
    raise RuntimeError("schedule case order drifted")
if len(schedule_records) != 12:
    raise RuntimeError("schedule must contain exactly 12 requests")

schedule_document = {
    "schema_version": "1.0.0",
    "status": "QUALIFIED_NOT_STARTED",
    "authorization_fingerprint": AUTHORIZATION_FINGERPRINT,
    "authorization_merge_commit": AUTHORIZATION_MERGE_COMMIT,
    "notebook_sha256": SOURCE_NOTEBOOK_SHA256,
    "case_manifest_sha256": CASE_MANIFEST_SHA256,
    "evaluation_plan_sha256": EVALUATION_PLAN_SHA256,
    "prompt_policy_sha256": PROMPT_POLICY_SHA256,
    "action_schema_sha256": ACTION_SCHEMA_SHA256,
    "request_count": 12,
    "worker_id": "worker_1",
    "cache_measurement_in_scope": False,
    "cache_claims_permitted": False,
    "hidden_retry_count": 0,
    "repair_attempt_count": 0,
    "replacement_request_count": 0,
    "full_measured_rerun_authorized": False,
    "requests": schedule_records,
}
(EVIDENCE_DIR / authorization.evidence.schedule_filename).write_text(
    canonical_json(schedule_document) + "\n",
    encoding="utf-8",
)

initial_checkpoint = {
    "schema_version": "1.0.0",
    "status": "not_started",
    "completed_request_count": 0,
    "semantic_failure_count": 0,
    "infrastructure_failure_count": 0,
    "cleanup_status": "not_started",
    "updated_at": datetime.now(UTC).isoformat(),
}
(EVIDENCE_DIR / authorization.evidence.checkpoint_filename).write_text(
    canonical_json(initial_checkpoint) + "\n",
    encoding="utf-8",
)

EXECUTION_PREFLIGHT_QUALIFIED = True
print(
    canonical_json(
        {
            "status": "AUTHORIZED_TO_EXECUTE_BOUNDED_CANARY",
            "authorization_fingerprint": AUTHORIZATION_FINGERPRINT,
            "authorization_merge_commit": AUTHORIZATION_MERGE_COMMIT,
            "notebook_sha256": SOURCE_NOTEBOOK_SHA256,
            "request_count": 12,
            "worker_id": "worker_1",
            "cache_measurement_in_scope": False,
            "full_measured_rerun_authorized": False,
        }
    )
)

In [ ]:
# Cell 09 — Worker Lifecycle and One-Shot HTTP Transport


def utc_now() -> str:
    return datetime.now(UTC).isoformat()


def port_is_open(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as client:
        client.settimeout(0.5)
        return client.connect_ex(("127.0.0.1", port)) == 0


def http_get_json(url: str, timeout_seconds: int = 5) -> dict[str, Any]:
    with urllib.request.urlopen(url, timeout=timeout_seconds) as response:
        payload = json.loads(response.read().decode("utf-8"))
    if not isinstance(payload, dict):
        raise RuntimeError("HTTP response must be one JSON object")
    return payload


def wait_for_worker(process: subprocess.Popen[str], timeout_seconds: int = 240) -> None:
    deadline = time.monotonic() + timeout_seconds
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError("worker exited before health qualification")
        try:
            with urllib.request.urlopen(
                f"http://127.0.0.1:{WORKER_PORT}/health",
                timeout=2,
            ) as response:
                health_status = response.status
            models = http_get_json(
                f"http://127.0.0.1:{WORKER_PORT}/v1/models",
                timeout_seconds=3,
            )
            model_ids = [item["id"] for item in models.get("data", [])]
            if health_status == 200 and model_ids == [SERVED_MODEL_NAME]:
                return
        except (
            OSError,
            urllib.error.URLError,
            json.JSONDecodeError,
            KeyError,
            TypeError,
        ):
            pass
        time.sleep(2)
    raise TimeoutError("worker_1 did not become healthy")


def start_worker() -> tuple[subprocess.Popen[str], Any]:
    if not EXECUTION_PREFLIGHT_QUALIFIED:
        raise RuntimeError("execution preflight must pass before worker start")
    if port_is_open(WORKER_PORT):
        raise RuntimeError("worker port is already open before authorized start")

    log_handle = WORKER_LOG.open("w", encoding="utf-8")
    environment = os.environ.copy()
    environment["CUDA_VISIBLE_DEVICES"] = "0"
    environment["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
    command = [
        sys.executable,
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        str(MODEL_DIR),
        "--tokenizer",
        str(MODEL_DIR),
        "--host",
        "127.0.0.1",
        "--port",
        str(WORKER_PORT),
        "--dtype",
        "half",
        "--seed",
        "7",
        "--max-model-len",
        "4096",
        "--enforce-eager",
        "--served-model-name",
        SERVED_MODEL_NAME,
        "--generation-config",
        "vllm",
        "--attention-backend",
        "TRITON_ATTN",
        "--gpu-memory-utilization",
        "0.70",
        "--disable-uvicorn-access-log",
        "--max-log-len",
        "0",
    ]
    process = subprocess.Popen(
        command,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
        env=environment,
    )
    wait_for_worker(process)
    return process, log_handle


def stop_worker(
    process: subprocess.Popen[str] | None,
    log_handle: Any | None,
) -> dict[str, object]:
    signal_path: list[str] = []
    return_code: int | None = None
    if process is not None:
        if process.poll() is None:
            process.send_signal(signal.SIGINT)
            signal_path.append("SIGINT")
            try:
                process.wait(timeout=45)
            except subprocess.TimeoutExpired:
                process.terminate()
                signal_path.append("SIGTERM")
                try:
                    process.wait(timeout=20)
                except subprocess.TimeoutExpired:
                    process.kill()
                    signal_path.append("SIGKILL")
                    process.wait(timeout=10)
        return_code = process.returncode
    if log_handle is not None:
        log_handle.close()

    deadline = time.monotonic() + 30
    while port_is_open(WORKER_PORT) and time.monotonic() < deadline:
        time.sleep(1)
    port_closed = not port_is_open(WORKER_PORT)
    return {
        "worker_id": "worker_1",
        "return_code": return_code,
        "signal_path": signal_path,
        "port": WORKER_PORT,
        "port_closed": port_closed,
        "status": "CLEAN" if port_closed else "FAILED",
    }


def post_json_once(
    *,
    payload: dict[str, object],
    timeout_seconds: int = 120,
) -> tuple[int, dict[str, Any], float]:
    body = canonical_json(payload).encode("utf-8")
    request = urllib.request.Request(
        f"http://127.0.0.1:{WORKER_PORT}/v1/chat/completions",
        data=body,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    started = time.perf_counter()
    try:
        with urllib.request.urlopen(
            request,
            timeout=timeout_seconds,
        ) as response:
            status = response.status
            response_payload = json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as error:
        raise RuntimeError(f"HTTP_{error.code}") from error
    elapsed_ms = (time.perf_counter() - started) * 1000.0
    if not isinstance(response_payload, dict):
        raise RuntimeError("chat-completion response must be one JSON object")
    return status, response_payload, elapsed_ms


print("worker lifecycle and one-shot transport ready")

In [ ]:
# Cell 10 — Twelve-Request Canary and Repository Scoring


def write_json(path: Path, payload: object) -> None:
    path.write_text(canonical_json(payload) + "\n", encoding="utf-8")


def write_ledger(path: Path, records: list[dict[str, object]]) -> None:
    text = "".join(canonical_json(record) + "\n" for record in records)
    path.write_text(text, encoding="utf-8")


def safe_failure_detail(error: BaseException) -> dict[str, str]:
    return {
        "exception_type": type(error).__name__,
        "safe_detail": "infrastructure boundary failed",
    }


def run_bounded_action_extraction_canary() -> dict[str, object]:
    if not EXECUTION_PREFLIGHT_QUALIFIED:
        raise RuntimeError("all preflight gates must pass before execution")

    ledger_path = EVIDENCE_DIR / authorization.evidence.ledger_filename
    checkpoint_path = EVIDENCE_DIR / authorization.evidence.checkpoint_filename
    evaluation_path = EVIDENCE_DIR / authorization.evidence.evaluation_filename
    report_path = EVIDENCE_DIR / authorization.evidence.report_filename
    summary_path = EVIDENCE_DIR / authorization.evidence.summary_filename

    records: list[dict[str, object]] = []
    scores = []
    process: subprocess.Popen[str] | None = None
    log_handle: Any | None = None
    infrastructure_failure: dict[str, str] | None = None
    started_at = utc_now()

    try:
        process, log_handle = start_worker()

        for sequence, case in enumerate(manifest.accepted_cases, start=1):
            transient_prompt = render_reconcile_balance_extraction_prompt(case)
            request_body: dict[str, object] = {
                "model": SERVED_MODEL_NAME,
                "messages": [{"role": "user", "content": transient_prompt}],
                "response_format": response_format,
                "temperature": float(authorization.decoding.temperature),
                "top_p": float(authorization.decoding.top_p),
                "seed": authorization.decoding.seed,
                "max_tokens": authorization.decoding.max_output_tokens,
                "stream": authorization.decoding.stream,
            }
            request_body_sha256 = hashlib.sha256(
                canonical_json(request_body).encode("utf-8")
            ).hexdigest()

            status_code, response_payload, elapsed_ms = post_json_once(payload=request_body)
            choices = response_payload.get("choices")
            usage = response_payload.get("usage")
            if status_code != 200 or not isinstance(choices, list):
                raise RuntimeError("chat-completion response envelope invalid")
            if len(choices) != 1 or not isinstance(usage, dict):
                raise RuntimeError("chat-completion response cardinality invalid")

            choice = choices[0]
            if not isinstance(choice, dict):
                raise RuntimeError("chat-completion choice must be an object")
            message = choice.get("message")
            if not isinstance(message, dict):
                raise RuntimeError("chat-completion message missing")
            output_text = message.get("content")
            finish_reason = choice.get("finish_reason")
            prompt_tokens = usage.get("prompt_tokens")
            completion_tokens = usage.get("completion_tokens")
            if not isinstance(output_text, str):
                raise RuntimeError("chat-completion output text missing")
            if finish_reason is not None and not isinstance(finish_reason, str):
                raise RuntimeError("chat-completion finish reason invalid")
            if not isinstance(prompt_tokens, int):
                raise RuntimeError("chat-completion prompt token count missing")
            if not isinstance(completion_tokens, int):
                raise RuntimeError("chat-completion completion token count missing")

            score = evaluate_reconcile_balance_extraction(
                case=case,
                output_text=output_text,
                finish_reason=finish_reason,
                completion_tokens=completion_tokens,
            )
            scores.append(score)
            record = {
                "schema_version": "1.0.0",
                "sequence": sequence,
                "eval_case_id": case.eval_case_id,
                "worker_id": "worker_1",
                "attempt_index": 1,
                "http_status": status_code,
                "elapsed_ms": round(elapsed_ms, 3),
                "request_body_sha256": request_body_sha256,
                "api_prompt_tokens": prompt_tokens,
                "api_completion_tokens": completion_tokens,
                "score": score.model_dump(mode="json"),
            }
            records.append(record)
            write_ledger(ledger_path, records)

            checkpoint = {
                "schema_version": "1.0.0",
                "status": "running",
                "completed_request_count": len(records),
                "semantic_failure_count": sum(
                    not item.first_attempt_task_success for item in scores
                ),
                "infrastructure_failure_count": 0,
                "cleanup_status": "pending",
                "updated_at": utc_now(),
            }
            write_json(checkpoint_path, checkpoint)

        if len(scores) != 12:
            raise RuntimeError("complete 12-record score set was not produced")
    except Exception as error:
        infrastructure_failure = safe_failure_detail(error)
    finally:
        cleanup = stop_worker(process, log_handle)

    completed_all_requests = len(scores) == 12 and infrastructure_failure is None
    report = None
    if completed_all_requests:
        report = build_action_extraction_evaluation_report(
            report_id="reconcile-balance-action-extraction-canary-report-v1",
            created_at=datetime.now(UTC),
            plan=plan,
            manifest=manifest,
            scores=tuple(scores),
        )
        write_json(evaluation_path, report.model_dump(mode="json"))
        final_status = (
            "ACTION_EXTRACTION_CANARY_PASSED"
            if report.gate_decision.value == "passed"
            else "ACTION_EXTRACTION_CANARY_FAILED_DIAGNOSTIC"
        )
    else:
        partial_evaluation = {
            "schema_version": "1.0.0",
            "status": "INFRASTRUCTURE_ABORTED",
            "completed_request_count": len(scores),
            "required_request_count": 12,
            "infrastructure_failure": infrastructure_failure,
            "full_measured_rerun_authorized": False,
        }
        write_json(evaluation_path, partial_evaluation)
        final_status = "ACTION_EXTRACTION_CANARY_INFRASTRUCTURE_ABORTED"

    checkpoint = {
        "schema_version": "1.0.0",
        "status": ("completed" if completed_all_requests else "infrastructure_aborted"),
        "completed_request_count": len(scores),
        "semantic_failure_count": sum(not item.first_attempt_task_success for item in scores),
        "infrastructure_failure_count": (0 if infrastructure_failure is None else 1),
        "infrastructure_failure": infrastructure_failure,
        "cleanup_status": cleanup["status"],
        "updated_at": utc_now(),
    }
    write_json(checkpoint_path, checkpoint)

    report_envelope: dict[str, object] = {
        "schema_version": "1.0.0",
        "status": final_status,
        "started_at": started_at,
        "completed_at": utc_now(),
        "repository_commit": AUTHORIZATION_MERGE_COMMIT,
        "authorization_fingerprint": AUTHORIZATION_FINGERPRINT,
        "notebook_sha256": SOURCE_NOTEBOOK_SHA256,
        "request_count": len(scores),
        "required_request_count": 12,
        "worker_id": "worker_1",
        "cleanup": cleanup,
        "infrastructure_failure": infrastructure_failure,
        "evaluation_report_sha256": (report.fingerprint() if report is not None else None),
        "gate_decision": (report.gate_decision.value if report is not None else None),
        "failed_case_ids": (list(report.failed_case_ids) if report is not None else []),
        "failure_code_counts": (report.failure_code_counts if report is not None else {}),
        "cache_measurement_in_scope": False,
        "cache_claims_permitted": False,
        "full_measured_rerun_authorized": False,
        "external_spend": "0",
        "customer_data_used": False,
        "raw_prompt_retained": False,
        "raw_output_retained": False,
        "raw_action_retained": False,
    }
    write_json(report_path, report_envelope)

    summary_lines = [
        f"status={final_status}",
        f"completed_request_count={len(scores)}",
        "required_request_count=12",
        f"semantic_failure_count={sum(not item.first_attempt_task_success for item in scores)}",
        f"infrastructure_failure_count={0 if infrastructure_failure is None else 1}",
        f"cleanup_status={cleanup['status']}",
        f"authorization_fingerprint={AUTHORIZATION_FINGERPRINT}",
        f"notebook_sha256={SOURCE_NOTEBOOK_SHA256}",
        "cache_measurement_in_scope=false",
        "full_measured_rerun_authorized=false",
        "external_spend=0",
        "customer_data_used=false",
    ]
    summary_path.write_text("\n".join(summary_lines) + "\n", encoding="utf-8")

    return {
        "status": final_status,
        "scores": scores,
        "records": records,
        "report": report,
        "report_envelope": report_envelope,
        "checkpoint": checkpoint,
        "cleanup": cleanup,
    }


RUN_RESULT = run_bounded_action_extraction_canary()
print(
    canonical_json(
        {
            "status": RUN_RESULT["status"],
            "completed_request_count": len(RUN_RESULT["scores"]),
            "semantic_failure_count": sum(
                not item.first_attempt_task_success for item in RUN_RESULT["scores"]
            ),
            "cleanup_status": RUN_RESULT["cleanup"]["status"],
        }
    )
)

In [ ]:
# Cell 11 — Privacy Scan and Evidence Packaging

FORBIDDEN_EVIDENCE_KEYS = {
    "messages",
    "content",
    "raw_prompt",
    "raw_output",
    "raw_action",
    "token_ids",
    "api_key",
    "authorization_header",
}


def scan_forbidden_keys(value: object, path: str = "$") -> list[str]:
    findings: list[str] = []
    if isinstance(value, dict):
        for key, nested in value.items():
            if key in FORBIDDEN_EVIDENCE_KEYS:
                findings.append(f"{path}.{key}")
            findings.extend(scan_forbidden_keys(nested, f"{path}.{key}"))
    elif isinstance(value, list):
        for index, nested in enumerate(value):
            findings.extend(scan_forbidden_keys(nested, f"{path}[{index}]"))
    return findings


required_filenames = {
    authorization.evidence.schedule_filename,
    authorization.evidence.ledger_filename,
    authorization.evidence.checkpoint_filename,
    authorization.evidence.evaluation_filename,
    authorization.evidence.report_filename,
    authorization.evidence.summary_filename,
    authorization.evidence.model_snapshot_filename,
    authorization.evidence.worker_log_filename,
}
observed_filenames = {path.name for path in EVIDENCE_DIR.iterdir() if path.is_file()}
missing_filenames = required_filenames - observed_filenames
if missing_filenames:
    raise RuntimeError(f"required evidence files missing: {sorted(missing_filenames)}")

privacy_findings: dict[str, list[str]] = {}
for path in sorted(EVIDENCE_DIR.glob("*.json")):
    payload = json.loads(path.read_text(encoding="utf-8"))
    findings = scan_forbidden_keys(payload)
    if findings:
        privacy_findings[path.name] = findings

ledger_findings: list[str] = []
ledger_path = EVIDENCE_DIR / authorization.evidence.ledger_filename
for line_number, line in enumerate(
    ledger_path.read_text(encoding="utf-8").splitlines(),
    start=1,
):
    if not line:
        continue
    findings = scan_forbidden_keys(json.loads(line))
    ledger_findings.extend(f"line {line_number}: {finding}" for finding in findings)
if ledger_findings:
    privacy_findings[ledger_path.name] = ledger_findings

worker_log_text = WORKER_LOG.read_text(
    encoding="utf-8",
    errors="replace",
)
log_patterns = {
    "authorization_header": re.compile(r"authorization\s*:", re.IGNORECASE),
    "bearer_token": re.compile(
        r"\bbearer\s+[A-Za-z0-9._-]+",
        re.IGNORECASE,
    ),
    "api_key": re.compile(r"api[_-]?key", re.IGNORECASE),
}
log_findings = [name for name, pattern in log_patterns.items() if pattern.search(worker_log_text)]
for case in manifest.accepted_cases:
    if case.user_prompt in worker_log_text:
        log_findings.append(f"raw_case_prompt:{case.eval_case_id}")
if log_findings:
    privacy_findings[WORKER_LOG.name] = sorted(set(log_findings))

if privacy_findings:
    raise RuntimeError(f"privacy scan failed: {canonical_json(privacy_findings)}")

if EVIDENCE_ARCHIVE.exists():
    EVIDENCE_ARCHIVE.unlink()
with zipfile.ZipFile(
    EVIDENCE_ARCHIVE,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(EVIDENCE_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

with zipfile.ZipFile(EVIDENCE_ARCHIVE) as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"evidence ZIP integrity failed at {bad_member}")

print(
    canonical_json(
        {
            "status": RUN_RESULT["status"],
            "evidence_archive": EVIDENCE_ARCHIVE.name,
            "evidence_archive_sha256": sha256_file(EVIDENCE_ARCHIVE),
            "evidence_file_count": len(required_filenames),
            "privacy_scan": "passed",
            "zip_integrity": "passed",
            "cache_claims_permitted": False,
            "full_measured_rerun_authorized": False,
        }
    )
)

In [ ]:
# Cell 12 — Final Safe Summary

report = RUN_RESULT["report"]
safe_summary: dict[str, object] = {
    "status": RUN_RESULT["status"],
    "completed_request_count": len(RUN_RESULT["scores"]),
    "required_request_count": 12,
    "cleanup_status": RUN_RESULT["cleanup"]["status"],
    "evidence_archive": str(EVIDENCE_ARCHIVE),
    "evidence_archive_sha256": sha256_file(EVIDENCE_ARCHIVE),
    "authorization_fingerprint": AUTHORIZATION_FINGERPRINT,
    "authorization_merge_commit": AUTHORIZATION_MERGE_COMMIT,
    "notebook_sha256": SOURCE_NOTEBOOK_SHA256,
    "cache_measurement_in_scope": False,
    "full_measured_rerun_authorized": False,
}

if report is not None:
    safe_summary.update(
        {
            "gate_decision": report.gate_decision.value,
            "action_json_valid_rate": str(report.action_json_valid.rate),
            "action_schema_valid_rate": str(report.action_schema_valid.rate),
            "identity_accuracy": str(report.identity_accuracy.rate),
            "operand_accuracy": str(report.operand_accuracy.rate),
            "execution_success_rate": str(report.execution_success.rate),
            "final_answer_accuracy": str(report.final_answer_accuracy.rate),
            "first_attempt_task_success_rate": str(report.first_attempt_task_success.rate),
            "failed_case_ids": list(report.failed_case_ids),
            "failure_code_counts": report.failure_code_counts,
        }
    )

print(canonical_json(safe_summary))